# Clustering Analysis: PCA and t-SNE Visualization

This notebook demonstrates how to use the clustering module to analyze embedding spaces from the baseline face recognition model. We'll extract embeddings, apply PCA and t-SNE for dimensionality reduction, and visualize the identity separation.

## 1. Setup and Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import torch
from torch.utils.data import DataLoader
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Configure plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {pd.__version__}")

## 2. Load Embeddings and Identity Metadata

We'll import the clustering utilities from the evaluation module and demonstrate how to extract embeddings from a trained baseline model.

In [ ]:
# Import clustering utilities from src
sys.path.insert(0, str(Path.cwd()))

from src.evaluation.clustering import extract_embeddings, apply_pca, apply_tsne, plot_2d_embeddings, plot_pca_explained_variance
from src.datasets.face_dataset import CasiaFaceDataset, build_train_label_mapping
from src.models.resnet18 import build_baseline_resnet18
from src.utils.config import load_yaml_config, resolve_repo_path

# Configuration parameters (modify these based on your checkpoint location)
CONFIG_PATH = "experiments/configs/base.yaml"
CHECKPOINT_PATH = "experiments/runs/baseline_run/checkpoints/best.pt"  # Update with actual checkpoint path
SPLIT = "test"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64

print(f"Device: {DEVICE}")
print(f"Config: {CONFIG_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# Load configuration
config = load_yaml_config(CONFIG_PATH)
data_root = resolve_repo_path(config["data"]["root_dir"])
split_file = resolve_repo_path(config["data"]["split_file"])

print(f"Data root: {data_root}")
print(f"Split file: {split_file}")

# Load label mapping from training split
label_mapping = build_train_label_mapping(split_file)
num_classes = len(label_mapping)
print(f"Number of training classes: {num_classes}")

In [ ]:
# Build and load model
model_cfg = config["model"]
model = build_baseline_resnet18(
    pretrained=model_cfg.get("pretrained", True),
    embedding_dim=int(model_cfg.get("embedding_dim", 512)),
    normalize_embeddings=model_cfg.get("normalize_embeddings", True),
    classifier_num_classes=num_classes,
)
model = model.to(DEVICE)

# Load checkpoint
checkpoint_path = resolve_repo_path(CHECKPOINT_PATH)
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded checkpoint from: {checkpoint_path}")
else:
    print(f"WARNING: Checkpoint not found at {checkpoint_path}")
    print("Using untrained model for demonstration.")

In [ ]:
# Load dataset
dataset = CasiaFaceDataset(
    data_root=data_root,
    split_file=split_file,
    split_name=SPLIT,
    image_size=int(config["data"].get("image_size", 224)),
    train=False,
    label_mapping=label_mapping,
)
print(f"Dataset size: {len(dataset)} samples")

# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)
print(f"DataLoader created with batch size: {BATCH_SIZE}")

In [ ]:
# Extract embeddings
print("Extracting embeddings...")
embeddings, labels, _ = extract_embeddings(
    model,
    dataloader,
    device=torch.device(DEVICE),
    return_labels=True,
    amp_enabled=config["system"].get("amp", False),
)
embeddings_np = embeddings.numpy()
labels_np = labels.numpy()

print(f"Embeddings shape: {embeddings_np.shape}")
print(f"Labels shape: {labels_np.shape}")
print(f"Unique identities: {len(np.unique(labels_np))}")
print(f"Embedding dtype: {embeddings_np.dtype}")

## 3. Preprocessing: Filtering and Normalization

In [ ]:
# Optional: Sampling for visualization clarity
# Remove classes with very few samples and randomly sample identities for visualization
MIN_SAMPLES_PER_CLASS = 5
MAX_IDENTITIES_FOR_PLOT = 50  # Limit identities for clearer visualization

# Count samples per identity
unique_ids, counts = np.unique(labels_np, return_counts=True)
valid_ids = unique_ids[counts >= MIN_SAMPLES_PER_CLASS]

# Filter embeddings and labels
mask = np.isin(labels_np, valid_ids)
embeddings_filtered = embeddings_np[mask]
labels_filtered = labels_np[mask]

print(f"After filtering (min {MIN_SAMPLES_PER_CLASS} samples per class):")
print(f"  Embeddings shape: {embeddings_filtered.shape}")
print(f"  Unique identities: {len(np.unique(labels_filtered))}")

# Further sampling for visualization
if len(np.unique(labels_filtered)) > MAX_IDENTITIES_FOR_PLOT:
    sampled_ids = np.random.choice(np.unique(labels_filtered), MAX_IDENTITIES_FOR_PLOT, replace=False)
    mask_sampled = np.isin(labels_filtered, sampled_ids)
    embeddings_viz = embeddings_filtered[mask_sampled]
    labels_viz = labels_filtered[mask_sampled]
    print(f"Sampled to {MAX_IDENTITIES_FOR_PLOT} identities for visualization")
else:
    embeddings_viz = embeddings_filtered
    labels_viz = labels_filtered
    print(f"Using all {len(np.unique(labels_filtered))} identities")

print(f"Final embedding shape for visualization: {embeddings_viz.shape}")

## 4. Dimensionality Reduction: PCA

In [ ]:
# Apply PCA for 2D projection
print("Applying PCA...")
pca_2d, pca_obj = apply_pca(embeddings_viz, n_components=2)

# Print PCA variance information
print(f"PCA explained variance ratio: {pca_obj.explained_variance_ratio_}")
print(f"Cumulative variance (2 components): {np.cumsum(pca_obj.explained_variance_ratio_)[-1]:.4f}")

# Plot PCA explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance by component
all_pca_2d, all_pca_obj = apply_pca(embeddings_viz, n_components=50)
ax1.bar(range(1, 11), all_pca_obj.explained_variance_ratio_[:10])
ax1.set_xlabel("Principal Component", fontsize=12)
ax1.set_ylabel("Explained Variance Ratio", fontsize=12)
ax1.set_title("Explained Variance by Component", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3, axis="y")

# Cumulative explained variance
cumsum = np.cumsum(all_pca_obj.explained_variance_ratio_)
ax2.plot(range(1, len(cumsum) + 1), cumsum, marker="o", linestyle="-", linewidth=2)
ax2.axhline(y=0.95, color="r", linestyle="--", label="95% variance")
ax2.set_xlabel("Number of Components", fontsize=12)
ax2.set_ylabel("Cumulative Explained Variance", fontsize=12)
ax2.set_title("Cumulative Explained Variance", fontsize=14, fontweight="bold")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2D PCA projection
plot_2d_embeddings(
    pca_2d,
    labels_viz,
    title=f"PCA 2D Projection ({SPLIT} split)",
    save_path=None,  # Display instead of saving
    figsize=(14, 11),
    alpha=0.6,
    s=30,
)

## 5. Dimensionality Reduction: t-SNE with Hyperparameter Tuning

In [ ]:
# Apply t-SNE with default parameters
print("Applying t-SNE (this may take a minute...)...")
tsne_2d, tsne_obj = apply_tsne(
    embeddings_viz,
    n_components=2,
    perplexity=30.0,
    n_iter=1000,
    random_state=SEED,
)
print("t-SNE completed!")

In [ ]:
# Plot 2D t-SNE projection
plot_2d_embeddings(
    tsne_2d,
    labels_viz,
    title=f"t-SNE 2D Projection (perplexity=30, {SPLIT} split)",
    save_path=None,
    figsize=(14, 11),
    alpha=0.6,
    s=30,
)

## 6. Comparison: PCA vs t-SNE Side-by-Side

In [ ]:
# Create comparison figure
fig, axes = plt.subplots(1, 2, figsize=(20, 9))

# PCA projection
unique_labels = np.unique(labels_viz)
colors = plt.cm.hsv(np.linspace(0, 1, len(unique_labels)))
label_to_color = {label: colors[i] for i, label in enumerate(unique_labels)}

for label in unique_labels:
    mask = labels_viz == label
    axes[0].scatter(
        pca_2d[mask, 0],
        pca_2d[mask, 1],
        c=[label_to_color[label]],
        alpha=0.6,
        s=30,
        edgecolors="none",
    )

axes[0].set_xlabel("PC1", fontsize=12)
axes[0].set_ylabel("PC2", fontsize=12)
axes[0].set_title("PCA 2D Projection", fontsize=14, fontweight="bold")
axes[0].grid(True, alpha=0.3)

# t-SNE projection
for label in unique_labels:
    mask = labels_viz == label
    axes[1].scatter(
        tsne_2d[mask, 0],
        tsne_2d[mask, 1],
        c=[label_to_color[label]],
        alpha=0.6,
        s=30,
        edgecolors="none",
    )

axes[1].set_xlabel("t-SNE 1", fontsize=12)
axes[1].set_ylabel("t-SNE 2", fontsize=12)
axes[1].set_title("t-SNE 2D Projection (perplexity=30)", fontsize=14, fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nVisualization contains {len(unique_labels)} identities (identity IDs)")

## 7. Clustering Quality Metrics

In [ ]:
# Compute clustering metrics on original embeddings
print("Computing clustering quality metrics on original (full-dim) embeddings...\n")

# Silhouette Score (higher is better, range [-1, 1])
# Warning: can be slow for large datasets
if len(embeddings_viz) <= 5000:
    silhouette = silhouette_score(embeddings_viz, labels_viz)
    print(f"Silhouette Score: {silhouette:.4f}")
    print("  Interpretation: Score near 1 indicates well-separated clusters.")
    print("                  Score near 0 indicates overlapping clusters.")
    print("                  Score near -1 indicates misclassified points.\n")
else:
    print("Silhouette Score: Skipped (>5000 samples - too slow)\n")

# Davies-Bouldin Index (lower is better, min value 0)
db_index = davies_bouldin_score(embeddings_viz, labels_viz)
print(f"Davies-Bouldin Index: {db_index:.4f}")
print("  Interpretation: Lower values indicate better cluster separation.\n")

# Calinski-Harabasz Score (higher is better)
ch_score = calinski_harabasz_score(embeddings_viz, labels_viz)
print(f"Calinski-Harabasz Score: {ch_score:.4f}")
print("  Interpretation: Higher values indicate better-defined clusters.\n")

## 8. Summary and Interpretation

### What the visualizations tell us:

- **PCA (Linear method)**: Shows global structure. Identities that are linearly separable will form distinct clusters.
- **t-SNE (Non-linear method)**: Reveals local structure better. Even non-linearly separable identities will form tighter clusters if embeddings are well-learned.

### Key observations to look for:

1. **Well-trained model**: Identity clusters should be tight and well-separated.
2. **Poor training**: Clusters will be overlapping or dispersed.
3. **Dimensionality reduction quality**: If PCA and t-SNE show similar separation patterns, it suggests the embedding space has well-structured clusters in the high-dimensional space.

### Next steps:

1. Train the baseline model on your dataset
2. Run this clustering analysis on the trained model
3. Compare baseline clustering with metric learning approaches (triplet loss)

In [ ]:
# Optional: Save results to JSON for documentation
output_dir = Path("experiments/artifacts/clustering")
output_dir.mkdir(parents=True, exist_ok=True)

results_metadata = {
    "split": SPLIT,
    "num_samples": len(embeddings_viz),
    "num_identities": len(np.unique(labels_viz)),
    "embedding_dim": embeddings_viz.shape[1],
    "pca_explained_variance_first_2": pca_obj.explained_variance_ratio_.tolist(),
    "metrics": {
        "davies_bouldin_score": float(db_index),
        "calinski_harabasz_score": float(ch_score),
    },
}

# Add silhouette if computed
if 'silhouette' in locals():
    results_metadata["metrics"]["silhouette_score"] = float(silhouette)

# Save metadata
metadata_file = output_dir / "clustering_metadata.json"
with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(results_metadata, f, indent=2)

print(f"Results saved to {metadata_file}")